In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [ ]:
print(train.shape)
print(test.shape)
print(train.head())
print(train.info())
print(train.describe())

In [ ]:
missing = train.isnull().sum()
missing_percent = (missing / len(train)) * 100
table = pd.concat([missing, missing_percent], axis=1)
table.columns = ["Missing", "Percent"]
table = table[table["Missing"] > 0].sort_values("Percent", ascending=False)
print(table.head(20))


In [ ]:
plt.figure(figsize=(10,5))
table.head(20)["Percent"].plot(kind="barh")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(train["SalePrice"], bins=50)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(np.log1p(train["SalePrice"]), bins=50)
plt.show()


In [ ]:
print(train["SalePrice"].mean())
print(train["SalePrice"].median())
print(train["SalePrice"].min())
print(train["SalePrice"].max())


In [ ]:
num = train.select_dtypes(include=[np.number]).columns
corr = train[num].corr()
sale_corr = corr["SalePrice"].sort_values(ascending=False)
print(sale_corr.head(11))


In [ ]:
top = sale_corr.head(11).index
plt.figure(figsize=(8,6))
sns.heatmap(train[top].corr(), annot=True)
plt.show()

In [ ]:
features = ["OverallQual","GrLivArea","GarageCars","TotalBsmtSF"]
plt.figure(figsize=(10,8))
for i,f in enumerate(features):
    plt.subplot(2,2,i+1)
    plt.scatter(train[f], train["SalePrice"])
plt.show()

In [ ]:
y = train["SalePrice"]
train_id = train["Id"]
test_id = test["Id"]

In [ ]:
train = train.drop(["Id","SalePrice"], axis=1)
test = test.drop(["Id"], axis=1)

In [ ]:
data = pd.concat([train,test], axis=0)
print(data.shape)

In [ ]:
cat = data.select_dtypes(include=["object"]).columns
for c in cat:
    data[c] = data[c].fillna("None")

num = data.select_dtypes(include=[np.number]).columns
for c in num:
    data[c] = data[c].fillna(data[c].median())

print(data.isnull().sum().sum())


In [ ]:
data["TotalSF"] = data["TotalBsmtSF"] + data["1stFlrSF"] + data["2ndFlrSF"]
data["TotalBath"] = data["FullBath"] + 0.5*data["HalfBath"] + data["BsmtFullBath"] + 0.5*data["BsmtHalfBath"]
data["TotalPorchSF"] = data["OpenPorchSF"] + data["EnclosedPorch"] + data["3SsnPorch"] + data["ScreenPorch"]

cat = data.select_dtypes(include=["object"]).columns
encoders = {}
for c in cat:
    le = LabelEncoder()
    data[c] = le.fit_transform(data[c])
    encoders[c] = le



In [ ]:
X_train = data[:len(train)]
X_test = data[len(train):]

print(X_train.shape)
print(X_test.shape)

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.2, random_state=42)


In [ ]:
def check(model, name):
    model.fit(X_tr, y_tr)
    p1 = model.predict(X_tr)
    p2 = model.predict(X_val)
    mae1 = mean_absolute_error(y_tr, p1)
    mae2 = mean_absolute_error(y_val, p2)
    rmse1 = np.sqrt(mean_squared_error(y_tr, p1))
    rmse2 = np.sqrt(mean_squared_error(y_val, p2))
    r21 = r2_score(y_tr, p1)
    r22 = r2_score(y_val, p2)
    print(name)
    print(mae1, mae2)
    print(rmse1, rmse2)
    print(r21, r22)
    return model, rmse2

models = {}
results = {}

In [ ]:
m1, r1 = check(LinearRegression(), "Linear Regression")
models["Linear"] = m1
results["Linear"] = r1

m2, r2 = check(Ridge(alpha=10), "Ridge")
models["Ridge"] = m2
results["Ridge"] = r2

m3, r3 = check(Lasso(alpha=10), "Lasso")
models["Lasso"] = m3
results["Lasso"] = r3

m4, r4 = check(RandomForestRegressor(n_estimators=100, random_state=42), "Random Forest")
models["RF"] = m4
results["RF"] = r4

m5, r5 = check(GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42), "Gradient Boosting")
models["GB"] = m5
results["GB"] = r5

In [ ]:
res = pd.DataFrame(list(results.items()), columns=["Model","RMSE"])
res = res.sort_values("RMSE")
print(res)

plt.figure(figsize=(8,5))
plt.barh(res["Model"], res["RMSE"])
plt.show()

best_name = res.iloc[0]["Model"]
best = models[best_name]

In [ ]:
if hasattr(best, "feature_importances_"):
    imp = pd.DataFrame({"Feature":X_train.columns,"Importance":best.feature_importances_})
    imp = imp.sort_values("Importance", ascending=False)
    print(imp.head(20))
    plt.figure(figsize=(8,6))
    plt.barh(imp.head(20)["Feature"], imp.head(20)["Importance"])
    plt.gca().invert_yaxis()
    plt.show()

In [ ]:

best.fit(X_train, y)
pred = best.predict(X_test)

sub = pd.DataFrame({"Id":test_id,"SalePrice":pred})
sub.to_csv("submission.csv", index=False)

print(sub["SalePrice"].describe())

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(pred, bins=50)
plt.show()